# Calendar Compression and Feature Engineering

In [ ]:
from pathlib import Path
import os
import requests
import zipfile

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

## 1. Obiettivo e strategia


- statistiche di occupazione mensili,
- statistiche di prezzo e prezzo corretto,
- statistiche sulle notti minime e massime,
- indicatori di stabilità della disponibilità nel tempo,
- feature di stagionalità e recenza.

Queste feature sono pensate per essere riusabili quando in futuro aggiungerai altri dati o altri repository città.

In [ ]:
# Full calendar fetch
domain = "datasets.techmatrix.it/airml"
token = "DI_xeno_2026"

cities = ["sicilia", "trentino", "venezia", "roma", "puglia",
          "napoli", "firenze", "milano", "bergamo", "bologna"]

for city in cities:
    data_dir = os.path.join("./calendar/in", city)
    if os.path.exists(data_dir):
        print(f"Skipping {city}: folder already exists")
        continue
    
    url = f"https://{domain}/calendar/{city}.zip?token={token}"
    resp = requests.get(url, stream=True)
    if resp.ok:
        os.makedirs(data_dir, exist_ok=True)
        zip_path = os.path.join("./calendar/in", f"{city}.zip")
        with open(zip_path, "wb") as f:
            for chunk in resp.iter_content(8192):
                if chunk:
                    f.write(chunk)
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(path=data_dir)
        os.remove(zip_path)
    else:
        print(f"Failed to download {city}: {resp.status_code}")

## 2. Creazione di feature di calendario

In questa sezione estraiamo solo le feature di calendario davvero essenziali, per tenere il file compatto e ridurre la ridondanza.

- `occ_days_m`: numero di giorni occupati nel mese `m`, calcolato come somma di `available == 'f'` per ogni listing e mese.
- `observed_days_m`: numero totale di giorni osservati nel mese `m`, utile per normalizzare le frequenze.
- `occupancy_rate_m`: quota di occupazione del mese `m`, definita come `occ_days_m / observed_days_m`.
- `total_occ_days`: giorni occupati totali sull'intero orizzonte osservato.
- `total_observed_days`: giorni complessivamente osservati per il listing.
- `monthly_occupancy_mean`: media delle occupazioni mensili, utile per sintetizzare il comportamento medio del listing.

Essendo il file molto grandi, leggiamo i `calendar.csv` a chunk, puliamo `listing_id`, `date` e `available`, poi raggruppiamo prima per `listing_id` e mese e infine compattiamo il risultato a livello di listing.

In [ ]:
CALENDAR_DIR = Path('calendar')
calendar_files = sorted(CALENDAR_DIR.rglob('calendar.csv'))

if not calendar_files:
    raise FileNotFoundError('Nessun file calendar.csv trovato')

USECOLS = ['listing_id', 'date', 'available']
CHUNKSIZE = 200_000

Definiamo la funzione per processare ogni file calendar.csv, estraendo le feature mensili per ogni listing.

In [ ]:
def process_calendar_file(path, chunksize=CHUNKSIZE):
    # Nome della città dalla cartella
    city = path.parent.name
    monthly_parts = []

    # leggiamo il CSV a pezzi per gestire file molto grandi
    for chunk in pd.read_csv(path, usecols=USECOLS, dtype=str, keep_default_na=False, chunksize=chunksize):
        if chunk.empty:
            continue
        
        # Pulizia delle colonne: trimming e conversione tipi
        chunk['listing_id'] = chunk['listing_id'].str.strip()
        chunk['date'] = pd.to_datetime(chunk['date'], errors='coerce')

        # Filtro righe inutili: id vuoti, date non valide, valori disponibili non `t`/`f`
        chunk['available'] = chunk['available'].str.strip().str.lower()
        chunk = chunk[(chunk['listing_id'] != '') & chunk['date'].notna() & chunk['available'].isin(['t', 'f'])]
        if chunk.empty:
            continue
        
        # Estrai il mese e crea indicatori binari per occupato / disponibile
        chunk['month'] = chunk['date'].dt.month
        chunk['occupied'] = (chunk['available'] == 'f').astype('int8')
        chunk['is_available'] = (chunk['available'] == 't').astype('int8')

        # Aggregazione per listing_id + month: conta giorni occupati e giorni osservati
        monthly_parts.append(
            chunk.groupby(['listing_id', 'month'], as_index=False).agg(
                occ_days=('occupied', 'sum'),
                observed_days=('date', 'count'),
            )
        )

    # Se non abbiamo nessun dato valido, restituisci DataFrame vuoto
    if not monthly_parts:
        return pd.DataFrame()

    # Concateniamo i pezzi e sommiamo eventuali duplicati
    monthly = pd.concat(monthly_parts, ignore_index=True)
    monthly = monthly.groupby(['listing_id', 'month'], as_index=False).agg(
        occ_days=('occ_days', 'sum'),
        observed_days=('observed_days', 'sum'),
    )

    # Tasso di occupazione mensile (NaN quando observed_days == 0)
    monthly['occupancy_rate'] = monthly['occ_days'] / monthly['observed_days'].replace(0, np.nan)

    # Pivot: da righe per mese a colonne wide (occ_days_01 .. occupancy_rate_12)
    monthly_wide = monthly.pivot(index='listing_id', columns='month', values=['occ_days', 'occupancy_rate'])
    monthly_wide.columns = [f'{metric}_{month:02d}' for metric, month in monthly_wide.columns]
    monthly_wide = monthly_wide.reset_index()

    # Metriche aggregate per listing su tutto l'orizzonte osservato
    overall = monthly.groupby('listing_id', as_index=False).agg(
        total_occ_days=('occ_days', 'sum'),
        total_observed_days=('observed_days', 'sum'),
        monthly_occupancy_mean=('occupancy_rate', 'mean'),
    )

    # Occupancy_rate complessiva = tot occupati / tot osservati
    overall['occupancy_rate'] = overall['total_occ_days'] / overall['total_observed_days'].replace(0, np.nan)

    # Unisci e aggiungi la colonna city all'inizio
    city_df = monthly_wide.merge(overall, on='listing_id', how='left')
    city_df.insert(0, 'city', city)
    return city_df

Eseguiamo la funzione su tutti i file calendar.csv, ottenendo un DataFrame con le feature di calendario per ogni listing.

In [ ]:
city_compressed_frames = []
for path in calendar_files:
    print(f'Processing {path.parent.name}...')
    city_df = process_calendar_file(path)
    if not city_df.empty:
        city_compressed_frames.append(city_df)

calendar_compressed = pd.concat(city_compressed_frames, ignore_index=True)
calendar_compressed = calendar_compressed.sort_values(['city', 'listing_id']).reset_index(drop=True)
calendar_compressed.head()

## 3. Esportazione del dataset di calendario pre-elaborato

In [ ]:
OUTPUT_DIR = Path('calendar') / 'out'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

grouped = list(calendar_compressed.groupby("city"))
for city, city_df in grouped:
    city_dir = OUTPUT_DIR / city
    city_dir.mkdir(parents=True, exist_ok=True)
    out_path = city_dir / 'calendar.csv'
    city_df.drop(columns=['city']).to_csv(out_path, index=False)

print(f"Calendar compressed shape: {calendar_compressed.shape}")
print(f"Exported calendar data in folders under '{OUTPUT_DIR}'")